In [4]:
import pandas as pd

# 1. Load the two datasets
df_abund = pd.read_csv('/Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/results/inputs/single_taxon_abundance/single_taxon_abundance_Viruses_raw.csv')
df_canon = pd.read_csv('/Users/yarintamam/Downloads/canon.csv')

# 2. Merge the datasets
# We match the 'sample_name' from the abundance file with the 'library' column in the canon file
merged = pd.merge(df_abund, df_canon, left_on='sample_name', right_on='library', how='inner')

# 3. Pivot the table
# - index='subject' makes the subjects the rows
# - columns='day' makes the days the columns
# - values='relative_abundance' fills the table with the abundance values
# - aggfunc='mean' ensures that if a subject has multiple readings on the exact same day, 
#   they are averaged into a single value, guaranteeing each subject/day combination appears only once.
pivot_df = merged.pivot_table(index='subject', columns='day', values='relative_abundance', aggfunc='mean')

# 4. Save the resulting table to a new CSV file
output_file = 'subject_day_virus_abundance.csv'
pivot_df.to_csv(output_file)

In [6]:

# Condense the rows by removing NaNs and shifting data to the left
condensed_data = []
subjects = []

for subject, row in pivot_df.iterrows():
    # dropna() removes all NaN values, keeping chronological order
    values = row.dropna().tolist()
    condensed_data.append(values)
    subjects.append(subject)

# Create a new DataFrame from the lists
condensed_df = pd.DataFrame(condensed_data, index=subjects)

# Rename the columns sequentially
condensed_df.columns = [f"time_point_{i+1}" for i in range(condensed_df.shape[1])]
condensed_df.index.name = 'subject'

# Save to a new CSV file
output_file = 'condensed_virus_abundance.csv'
condensed_df.to_csv(output_file)